In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, DoubleType, IntegerType
from functools import reduce
from operator import or_
import time

t0 = time.time()

# ----------------------------------------------------------------
# 1. LEITURA UNICA da bronze
# ----------------------------------------------------------------
bronze = spark.read.table("vidaplus.bronze.fact_clima_diario")

# ----------------------------------------------------------------
# 2. CHECKS DE CONTRATO (carona na mesma passada de leitura)
#    Taxonomia (plano.md E1.8): objetivo -> QUARENTENA;
#    anomalo plausivel sem contrato (vento, pressao) -> passa sem check;
#    derivavel barato -> RECOMPUTE (contador de dias secos);
#    derivavel caro (estacao astronomica, epi week) -> MANTER do fornecedor.
# ----------------------------------------------------------------
COLS_TIPO = {  # colunas com contrato de tipo CHECADO na entrada
    # (dias_consecutivos_sem_chuva NAO entra: e RECOMPUTADA na silver — o valor
    # da fonte e apenas auditoria do recompute; lixo nele nao corrompe nada e
    # nao justifica quarentenar linha com medicoes validas)
    "data": DateType(),
    "temp_media_c": DoubleType(),
    "temp_max_c": DoubleType(),
    "temp_min_c": DoubleType(),
    "umidade_relativa_media_pct": DoubleType(),
    "precipitacao_mm": DoubleType(),
    "indice_uv_max": DoubleType(),
    "velocidade_vento_media_ms": DoubleType(),
    "pressao_atmosferica_hpa": DoubleType(),
    "ano_epidemiologico": IntegerType(),
    "semana_epidemiologica": IntegerType(),
}

# Regra 1: valor presente que nao converte p/ o tipo do contrato.
# String vazia NAO e violacao: e ausencia (nullif) — segue p/ silver como
# NULL e nao fura o calendario da serie com uma quarentena desnecessaria.
def _valor(c):
    """trim + vazio vira NULL (ausencia)"""
    return F.nullif(F.trim(c), F.lit(""))

cast_invalido = reduce(
    or_,
    [_valor(c).isNotNull() & _valor(c).try_cast(t).isNull() for c, t in COLS_TIPO.items()],
)

# Regra 2: fisicamente impossivel (max < media, media < min, chuva negativa)
# Nota: comparacao com NULL da NULL -> nao conta como violacao (3-valued logic)
t_max  = F.trim("temp_max_c").try_cast(DoubleType())
t_med  = F.trim("temp_media_c").try_cast(DoubleType())
t_min  = F.trim("temp_min_c").try_cast(DoubleType())
precip = F.trim("precipitacao_mm").try_cast(DoubleType())

fisica_impossivel = (t_max < t_med) | (t_med < t_min) | (precip < 0)

# Regra 3: fora do dominio DOCUMENTADO no dicionario (umidade 0-100, UV 0-14)
umidade = F.trim("umidade_relativa_media_pct").try_cast(DoubleType())
uv      = F.trim("indice_uv_max").try_cast(DoubleType())

fora_dominio_doc = ~umidade.between(0, 100) | ~uv.between(0, 14)

# Regra 0: chave obrigatoria do grao (data valida + municipio presente).
# Sem ela: data NULL/ilegivel escapava p/ silver (quebrando o orderBy do
# recompute) e cod NULL agrupava tudo na MESMA particao de grao, rotulado
# erradamente como grao_duplicado. Precedencia: chave > grao.
_data_chave = F.trim("data").try_cast(DateType())               # NULL se vazia/ilegivel
_cod_chave  = F.nullif(F.trim("cod_municipio_ibge"), F.lit(""))  # NULL se vazio
chave_ausente = _data_chave.isNull() | _cod_chave.isNull()

# Regra 4: grao duplicado (data x municipio) -> TODAS as linhas do grao quarentenam
# Particiona pelo valor CONVERTIDO (nao pela string crua): formatos
# diferentes do mesmo dia ("2021-03-15" vs "2021-3-15") nao escapam da regra.
w_grao = Window.partitionBy(_data_chave, _cod_chave)

checked = (
    bronze
    .withColumn("_chave_ausente", chave_ausente)
    .withColumn("_cnt_grao", F.count(F.lit(1)).over(w_grao))
    .withColumn("_cast_invalido", cast_invalido)
    .withColumn("_fisica_impossivel", fisica_impossivel)
    .withColumn("_fora_dominio_doc", fora_dominio_doc)
    .cache()
)

# NULL-safe (logica de Kleene): flag que avalia a NULL — ex. umidade NULL
# em ~between — NAO e violacao. Sem o coalesce, FALSE | NULL = NULL e a
# linha seria descartada por filter(~violou) E por filter(violou):
# perda silenciosa (silver + quarentena < bronze). Valor ausente segue
# para a silver como NULL (mesma politica da F2), nao para a quarentena.
violou = F.coalesce(
    F.col("_chave_ausente")
    | (F.col("_cnt_grao") > 1)
    | F.col("_cast_invalido")
    | F.col("_fisica_impossivel")
    | F.col("_fora_dominio_doc"),
    F.lit(False),
)

# ----------------------------------------------------------------
# 3. SILVER: tipos + trim + RECOMPUTE do contador de dias secos
#    (regra documentada: sequencia de dias com precipitacao <= 1 mm)
#    Nota: quarentenar linha do meio da serie cria buraco no calendario
#    da silver — o buraco fica auditavel na quarentena; imputacao e
#    decisao de gold, nao de silver.
# ----------------------------------------------------------------
tipados = checked.filter(~violou).select(
    F.trim("data").try_cast(DateType()).alias("data"),
    F.trim("cod_municipio_ibge").alias("cod_municipio_ibge"),  # nao-vazio garantido pela Regra 0
    _valor("municipio").alias("municipio"),  # texto: vazio -> NULL (ausencia uniforme na silver)
    _valor("uf").alias("uf"),
    F.trim("temp_media_c").try_cast(DoubleType()).alias("temp_media_c"),
    F.trim("temp_max_c").try_cast(DoubleType()).alias("temp_max_c"),
    F.trim("temp_min_c").try_cast(DoubleType()).alias("temp_min_c"),
    F.trim("umidade_relativa_media_pct").try_cast(DoubleType()).alias("umidade_relativa_media_pct"),
    F.trim("precipitacao_mm").try_cast(DoubleType()).alias("precipitacao_mm"),
    F.trim("indice_uv_max").try_cast(DoubleType()).alias("indice_uv_max"),
    F.trim("velocidade_vento_media_ms").try_cast(DoubleType()).alias("velocidade_vento_media_ms"),
    F.trim("pressao_atmosferica_hpa").try_cast(DoubleType()).alias("pressao_atmosferica_hpa"),
    _valor("estacao").alias("estacao"),   # derivavel-caro (astronomica): manter do fornecedor
    _valor("fonte").alias("fonte"),
    F.trim("ano_epidemiologico").try_cast(IntegerType()).alias("ano_epidemiologico"),
    F.trim("semana_epidemiologica").try_cast(IntegerType()).alias("semana_epidemiologica"),
    F.trim("dias_consecutivos_sem_chuva").try_cast(IntegerType()).alias("_dias_fonte"),  # auditoria do recompute
    F.col("_arquivo_origem"),
    F.col("_ingerido_em"),
)

# Recompute (gaps-and-islands): streak reinicia quando chove (> 1 mm).
# Conservadorismo coerente: precip NULL = dia chuvoso E buraco de
# calendario (dia ausente, ex. linha quarentenada) REINICIA o streak —
# dia 9 e dia 11 NAO sao consecutivos; estender o streak sobre um dia
# sem evidencia contradiz a propria regra do NULL.
# Nota: se o fornecedor contar ATRAVES do buraco, o n_div_recompute
# sinaliza o buraco (nao erro da fonte); idem o 1o dia seco de cada serie
# (fornecedor pode contar dias pre-janela da bronze).
w_dia = Window.partitionBy("cod_municipio_ibge").orderBy("data")
_buraco = F.datediff("data", F.lag("data").over(w_dia)) > 1  # NULL no 1o dia da serie

silver = (
    tipados
    .withColumn("_seco", F.coalesce((F.col("precipitacao_mm") <= 1).cast("int"), F.lit(0)))
    .withColumn(
        "_novo_streak",
        F.when(
            (F.col("_seco") == 1)
            & (
                (F.coalesce(F.lag("_seco").over(w_dia), F.lit(0)) == 0)
                | F.coalesce(_buraco, F.lit(True))
            ),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn("_streak_id", F.sum("_novo_streak").over(w_dia))
    .withColumn(
        "dias_consecutivos_sem_chuva",
        F.when(
            F.col("_seco") == 1,
            F.row_number().over(Window.partitionBy("cod_municipio_ibge", "_streak_id").orderBy("data")),
        ).otherwise(F.lit(0)),
    )
)

# Auditoria do recompute vs fornecedor (byproduct: valida a fonte a cada carga)
n_div_recompute = silver.filter(
    F.col("_dias_fonte").isNotNull()
    & (F.col("dias_consecutivos_sem_chuva") != F.col("_dias_fonte"))
).count()

silver = silver.drop("_seco", "_novo_streak", "_streak_id", "_dias_fonte")

silver.write.mode("overwrite").saveAsTable("vidaplus.silver.fact_clima_diario")

# ----------------------------------------------------------------
# 4. QUARENTENA: linhas que violaram regra (dado BRUTO, sem transformacao)
#    Precedencia da _regra_violada: chave > grao > cast > fisica > dominio
# ----------------------------------------------------------------
quarentena = (
    checked.filter(violou)
    .select(
        F.when(F.col("_chave_ausente"), F.lit("chave_obrigatoria_ausente"))
         .when(F.col("_cnt_grao") > 1, F.lit("grao_duplicado"))
         .when(F.col("_cast_invalido"), F.lit("cast_invalido"))
         .when(F.col("_fisica_impossivel"), F.lit("valor_fisicamente_impossivel"))
         .when(F.col("_fora_dominio_doc"), F.lit("fora_dominio_documentado"))
         .alias("_regra_violada"),
        F.col("_cnt_grao").alias("_total_no_grao"),
        *bronze.columns,
        F.current_timestamp().alias("_data_quarentena"),
    )
)

# silver.explain(mode="formatted")
quarentena.write.mode("overwrite").saveAsTable("vidaplus.silver.quarentena_clima_regras_violadas")

# ----------------------------------------------------------------
# 5. LOG da carga
# ----------------------------------------------------------------
n_bronze = bronze.count()
n_silver = silver.count()
n_quarentena = quarentena.count()

print(f"Bronze:      {n_bronze:>8,} linhas")
print(f"Silver:      {n_silver:>8,} linhas | divergencia recompute x fonte: {n_div_recompute:,}")
print(f"Quarentena:  {n_quarentena:>8,} linhas | taxa: {n_quarentena / n_bronze:.2%}")
print(f"Tempo total: {time.time() - t0:.1f}s")

checked.unpersist()  # libera o cache
